# <font color="#418FDE" size="6.5" uppercase>**Features and Splits**</font>

>Last update: 20260710.
    
By the end of this Lecture, you will be able to:
- Extract practical features from tabular, temporal, and image-style mechanical engineering data. 
- Implement manual train and test splitting without using machine learning libraries. 
- Scale numerical features and explain why scaling affects distance-based and gradient-based models. 


## **1. Mechanical Feature Extraction**

### **1.1. Practical Feature Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_02/Lecture_B/image_01_01.jpg?v=1783660085" width="250">



>* Turn raw data into meaningful features
>* Link features to physical machine behavior

>* Transform measurements into meaningful operating features
>* Clean units, missing values, and labels

>* Use engineer judgment to define meaningful features
>* Convert observations into comparable structured data



In [ ]:
#@title Python Code - Practical Feature Basics

# Practical features translate raw measurements.
# Mechanical examples include tables and signals.
# Small image grids can summarize defects.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create tiny pump test data.
pump_data = pd.DataFrame({
    "inlet_kpa": [101, 100, 102, 99],
    "outlet_kpa": [310, 355, 390, 420],
    "flow_lpm": [42, 48, 55, 61],
    "speed_rpm": [1450, 1750, 2050, 2350],
})

# Add physically meaningful tabular features.
pump_data["pressure_rise_kpa"] = pump_data["outlet_kpa"] - pump_data["inlet_kpa"]
pump_data["flow_fraction"] = pump_data["flow_lpm"] / 60.0
pump_data["speed_band"] = pd.cut(pump_data["speed_rpm"], bins=[0, 1600, 2200, 3000], labels=["low", "medium", "high"])

# Create a short vibration signal.
time_seconds = np.linspace(0.0, 1.0, 100)
vibration_g = 0.4 * np.sin(2 * np.pi * 12 * time_seconds)
vibration_g += 0.08 * np.sin(2 * np.pi * 36 * time_seconds)

# Extract simple temporal features.
rms_vibration = float(np.sqrt(np.mean(vibration_g ** 2)))
peak_vibration = float(np.max(np.abs(vibration_g)))
threshold_count = int(np.sum(np.abs(vibration_g) > 0.35))

# Create a tiny thermal image grid.
thermal_c = np.array([
    [32, 33, 34, 33, 32],
    [33, 36, 42, 37, 33],
    [34, 43, 78, 45, 34],
    [33, 37, 44, 38, 33],
    [32, 33, 34, 33, 32],
])

# Extract image-style hot spot features.
hot_mask = thermal_c > 50
hot_area_pixels = int(np.sum(hot_mask))
max_temperature = int(np.max(thermal_c))
mean_temperature = float(np.mean(thermal_c))

# Validate tiny example sizes.
assert pump_data.shape == (4, 7)
assert vibration_g.size == time_seconds.size
assert thermal_c.shape == (5, 5)

# Print compact teaching results.
print("Tabular feature columns:", list(pump_data.columns[-3:]))
print("Last pump pressure rise:", int(pump_data["pressure_rise_kpa"].iloc[-1]), "kPa")
print("Vibration RMS and peak:", round(rms_vibration, 3), round(peak_vibration, 3), "g")
print("Threshold exceedances:", threshold_count)
print("Hot area, max, mean:", hot_area_pixels, max_temperature, round(mean_temperature, 1))

# Plot the tiny thermal image.
plt.figure(figsize=(4, 3))
plt.imshow(thermal_c, cmap="inferno")
plt.colorbar(label="Temperature C")
plt.title("Thermal feature example")
plt.tight_layout()
plt.show()



### **1.2. Signal Features**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_02/Lecture_B/image_01_02.jpg?v=1783660087" width="250">



>* Summarize time signals into useful features
>* Preserve patterns relevant to machine behavior

>* Summarize signal level, extremes, and variability
>* Choose windows to capture desired behavior

>* Frequency patterns reveal machine behavior and faults
>* Useful features reflect real physical meaning



In [ ]:
#@title Python Code - Signal Features

# Signal features summarize changing sensor measurements.
# Small windows reveal practical machine behavior.
# Built-in calculations support engineering interpretation.

import math
import statistics
import numpy as np
import matplotlib.pyplot as plt

# Create a deterministic vibration signal.
np.random.seed(7)
sample_rate_hz = 200
seconds_recorded = 2
sample_count = sample_rate_hz * seconds_recorded

# Build time values for the sensor.
time_seconds = np.arange(sample_count) / sample_rate_hz
rotation_hz = 12
base_signal = 0.35 * np.sin(2 * math.pi * rotation_hz * time_seconds)

# Add noise and one impact event.
noise_signal = 0.05 * np.random.randn(sample_count)
impact_signal = np.zeros(sample_count)
impact_signal[180:185] = 0.9
vibration_g = base_signal + noise_signal + impact_signal

# Validate the signal before feature extraction.
assert vibration_g.size == sample_count
assert sample_rate_hz > 0
assert np.isfinite(vibration_g).all()

# Extract common time-domain signal features.
mean_g = float(np.mean(vibration_g))
std_g = float(np.std(vibration_g, ddof=1))
rms_g = float(np.sqrt(np.mean(vibration_g ** 2)))
peak_g = float(np.max(np.abs(vibration_g)))

# Count crossings through the central level.
centered_signal = vibration_g - mean_g
crossing_flags = centered_signal[:-1] * centered_signal[1:] < 0
zero_crossings = int(np.sum(crossing_flags))

# Estimate dominant frequency using a simple spectrum.
frequency_values = np.fft.rfftfreq(sample_count, d=1 / sample_rate_hz)
spectrum_values = np.abs(np.fft.rfft(vibration_g - mean_g))
spectrum_values[0] = 0
peak_index = int(np.argmax(spectrum_values))

# Store features in a compact dictionary.
features = {
    "mean_g": mean_g,
    "std_g": std_g,
    "rms_g": rms_g,
    "peak_g": peak_g,
    "zero_crossings": zero_crossings,
    "dominant_hz": float(frequency_values[peak_index]),
}

# Print a compact engineering feature summary.
print("Signal feature summary for a pump vibration window:")
for name, value in features.items():
    print(f"{name:15s}: {value:.3f}")

# Plot the signal and highlight the impact.
plt.figure(figsize=(8, 3))
plt.plot(time_seconds, vibration_g, linewidth=1.5, label="vibration")
plt.axhline(mean_g, color="black", linestyle="--", label="mean")
plt.xlabel("Time (seconds)")
plt.ylabel("Acceleration (g)")
plt.title("Signal features summarize a vibration time window")
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()



### **1.3. Derived Features**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_02/Lecture_B/image_01_03.jpg?v=1783660089" width="250">



>* Create meaningful variables from existing measurements
>* Use engineering judgment, not random combinations

>* Combine measurements into differences, ratios, and rates
>* Reveal mechanics of wear, loading, and efficiency

>* Convert image data into useful measurements
>* Avoid noisy, inconsistent, or future-leaking features



In [ ]:
#@title Python Code - Derived Features

# Derived features connect measurements to mechanics.
# Small examples keep engineering transformations clear.
# Built-in tools make feature logic transparent.

import math
import statistics as stats

# Store tiny pump test measurements.
pump = {
    "inlet_kpa": 120.0,
    "outlet_kpa": 355.0,
    "flow_lpm": 48.0,
    "speed_rpm": 1750.0,
}

# Derive pressure rise and normalized flow.
pressure_rise = pump["outlet_kpa"] - pump["inlet_kpa"]
flow_per_speed = pump["flow_lpm"] / pump["speed_rpm"]

# Store simple manufacturing measurements.
part = {
    "length_in": 4.02,
    "target_in": 4.00,
    "mass_g": 185.0,
    "cycle_s": 32.0,
}

# Derive deviation and production rate.
length_error = part["length_in"] - part["target_in"]
parts_per_hour = 3600.0 / part["cycle_s"]

# Store short temperature history.
times_s = [0, 10, 20, 30]
temps_c = [22.0, 28.0, 35.0, 43.0]

# Validate matching temporal list lengths.
assert len(times_s) == len(temps_c)
assert len(times_s) >= 2

# Derive average warm-up rate.
time_span = times_s[-1] - times_s[0]
temp_span = temps_c[-1] - temps_c[0]
warmup_rate = temp_span / time_span

# Store tiny thermal image temperatures.
thermal_c = [
    [41.0, 43.0, 45.0],
    [42.0, 58.0, 61.0],
    [40.0, 44.0, 46.0],
]

# Flatten pixels for simple image features.
pixels = [value for row in thermal_c for value in row]
hot_pixels = [value for value in pixels if value >= 55.0]

# Derive hot area and contrast.
hot_area_pixels = len(hot_pixels)
mean_hot = stats.mean(hot_pixels)
mean_all = stats.mean(pixels)
thermal_contrast = mean_hot - mean_all

# Print compact teaching results.
print("Derived mechanical features:")
print(f"Pump pressure rise: {pressure_rise:.1f} kPa")
print(f"Flow per speed: {flow_per_speed:.4f} L/min/rpm")
print(f"Length error: {length_error:.2f} in")
print(f"Production rate: {parts_per_hour:.1f} parts/hour")
print(f"Warm-up rate: {warmup_rate:.2f} C/s")
print(f"Hot area: {hot_area_pixels} pixels")
print(f"Thermal contrast: {thermal_contrast:.1f} C")



## **2. Manual Data Splitting**

### **2.1. Split Strategy**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_02/Lecture_B/image_02_01.jpg?v=1783660095" width="250">



>* Plan splits to test unseen data fairly
>* Match held-out data to real future use

>* Randomly split only truly independent rows
>* Group related data and balance rare outcomes

>* Preserve time order in temporal splits
>* Group related images and document assumptions



In [ ]:
#@title Python Code - Split Strategy

# Manual splitting protects honest model evaluation.
# Engineering rows often hide grouped relationships.
# This example compares two split strategies.

import random
import pandas as pd

# Create repeated measurements from several pumps.
random.seed(7)
rows = []

# Each pump contributes several related observations.
for pump_id in range(1, 7):
    base_vibration = 0.18 * pump_id
    for minute in range(4):
        rows.append({
            "pump_id": f"P{pump_id}",
            "minute": minute,
            "vibration_mm_s": round(base_vibration + random.random() * 0.05, 3),
            "temperature_F": round(145 + pump_id * 3 + random.random(), 1),
            "needs_service": pump_id >= 5,
        })

# Store the tiny engineering table.
data = pd.DataFrame(rows)
assert len(data) == 24

# Random row splitting can leak pump identity.
indices = list(data.index)
random.shuffle(indices)

# Use a simple eighty twenty split.
cut = int(0.8 * len(indices))
train_rows = data.loc[indices[:cut]]
test_rows = data.loc[indices[cut:]]

# Group splitting keeps pumps separated.
pumps = sorted(data["pump_id"].unique())
random.shuffle(pumps)

# Hold out complete pumps for testing.
test_pumps = set(pumps[-2:])
group_test = data[data["pump_id"].isin(test_pumps)]
group_train = data[~data["pump_id"].isin(test_pumps)]

# Summarize overlap and failure balance.
row_overlap = set(train_rows["pump_id"]) & set(test_rows["pump_id"])
group_overlap = set(group_train["pump_id"]) & set(group_test["pump_id"])

# Print concise teaching results.
print("Manual split strategy demonstration")
print(f"Rows: {len(data)}, pumps: {data['pump_id'].nunique()}")
print(f"Random row split sizes: {len(train_rows)} train, {len(test_rows)} test")
print(f"Random split shared pumps: {sorted(row_overlap)}")
print(f"Group split sizes: {len(group_train)} train, {len(group_test)} test")
print(f"Group split shared pumps: {sorted(group_overlap)}")
print(f"Group test pumps: {sorted(test_pumps)}")
print("Choose the split that matches future engineering use.")



### **2.2. Train Test Split**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_02/Lecture_B/image_02_02.jpg?v=1783660091" width="250">



>* Train learns patterns from selected data
>* Test checks generalization to unseen cases

>* Choose a reproducible train test proportion
>* Keep test data balanced and representative

>* Respect time order and group boundaries
>* Document split rules for reproducible evaluation



In [ ]:
#@title Python Code - Train Test Split

# Manual splitting supports fair model evaluation.
# This example uses pump vibration records.
# No machine learning libraries are needed.

import random
import pandas as pd

# Create a tiny engineering dataset.
records = [
    {"pump_id": "P01", "vibration_mm_s": 2.1, "temperature_C": 61, "failed": 0},
    {"pump_id": "P02", "vibration_mm_s": 2.8, "temperature_C": 64, "failed": 0},
    {"pump_id": "P03", "vibration_mm_s": 5.9, "temperature_C": 82, "failed": 1},

    {"pump_id": "P04", "vibration_mm_s": 3.0, "temperature_C": 66, "failed": 0},
    {"pump_id": "P05", "vibration_mm_s": 6.4, "temperature_C": 88, "failed": 1},
    {"pump_id": "P06", "vibration_mm_s": 2.4, "temperature_C": 63, "failed": 0},

    {"pump_id": "P07", "vibration_mm_s": 4.9, "temperature_C": 78, "failed": 1},
    {"pump_id": "P08", "vibration_mm_s": 2.6, "temperature_C": 65, "failed": 0},
    {"pump_id": "P09", "vibration_mm_s": 5.5, "temperature_C": 80, "failed": 1},

    {"pump_id": "P10", "vibration_mm_s": 3.2, "temperature_C": 67, "failed": 0},
    {"pump_id": "P11", "vibration_mm_s": 2.9, "temperature_C": 62, "failed": 0},
    {"pump_id": "P12", "vibration_mm_s": 6.1, "temperature_C": 85, "failed": 1},
]

# Convert records into a table.
df = pd.DataFrame(records)

# Shuffle row positions reproducibly.
random.seed(42)
indices = list(range(len(df)))
random.shuffle(indices)

# Choose an eighty percent training split.
train_size = int(0.8 * len(indices))
assert 0 < train_size < len(indices)

# Slice shuffled indices into two sets.
train_indices = indices[:train_size]
test_indices = indices[train_size:]

# Build separate training and test tables.
train_df = df.iloc[train_indices].reset_index(drop=True)
test_df = df.iloc[test_indices].reset_index(drop=True)

# Check that no row appears twice.
train_ids = set(train_df["pump_id"])
test_ids = set(test_df["pump_id"])
assert train_ids.isdisjoint(test_ids)

# Summarize the manual split.
print("Manual train test split for pump records")
print(f"Total rows: {len(df)}")
print(f"Training rows: {len(train_df)}")
print(f"Test rows: {len(test_df)}")

# Compare failure balance across splits.
train_rate = train_df["failed"].mean()
test_rate = test_df["failed"].mean()
print(f"Training failure rate: {train_rate:.2f}")
print(f"Test failure rate: {test_rate:.2f}")

# Show reserved test identifiers only.
print("Reserved test pumps:", sorted(test_ids))



### **2.3. Leakage Prevention**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_02/Lecture_B/image_02_03.jpg?v=1783660097" width="250">



>* Leakage makes test results falsely optimistic
>* Split before preprocessing to protect evaluation

>* Keep test data locked after splitting
>* Fit preprocessing using training data only

>* Split by machine, batch, specimen, or time
>* Preserve realistic, honest model evaluation



In [ ]:
#@title Python Code - Leakage Prevention

# Demonstrate leakage during manual splitting.
# Use only training statistics afterward.
# Keep the example small and clear.

import random
import statistics

# Create tiny pump sensor records.
random.seed(7)
records = []

# Simulate temperatures and vibration labels.
for pump_id in range(1, 13):
    temp_c = 55 + pump_id * 1.8
    vibration = 0.12 * temp_c + random.uniform(-1.0, 1.0)
    records.append((pump_id, temp_c, vibration))

# Shuffle before manual row splitting.
random.shuffle(records)
split_index = int(len(records) * 0.75)
train_rows = records[:split_index]
test_rows = records[split_index:]

# Validate split sizes before preprocessing.
assert len(train_rows) > 1 and len(test_rows) > 0
train_temps = [row[1] for row in train_rows]
test_temps = [row[1] for row in test_rows]

# This leaks test information into preprocessing.
all_temps = train_temps + test_temps
leaky_mean = statistics.mean(all_temps)
leaky_std = statistics.pstdev(all_temps)

# This uses only training information.
safe_mean = statistics.mean(train_temps)
safe_std = statistics.pstdev(train_temps)
assert safe_std > 0 and leaky_std > 0

# Apply both scalers to the locked test set.
first_test_temp = test_temps[0]
leaky_scaled = (first_test_temp - leaky_mean) / leaky_std
safe_scaled = (first_test_temp - safe_mean) / safe_std

# Print a compact leakage comparison.
print("Manual split sizes:", len(train_rows), "train,", len(test_rows), "test")
print("Leaky mean uses all rows:", round(leaky_mean, 2), "deg C")
print("Safe mean uses train only:", round(safe_mean, 2), "deg C")
print("First test temperature:", round(first_test_temp, 2), "deg C")
print("Leaky scaled test value:", round(leaky_scaled, 3))
print("Safe scaled test value:", round(safe_scaled, 3))
print("Rule: split first, then learn preprocessing from training only.")



## **3. Feature Scaling**

### **3.1. Scaling Fundamentals**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_02/Lecture_B/image_03_01.jpg?v=1783660099" width="250">



>* Make feature magnitudes comparable before modeling
>* Prevent units from misleading the model

>* Scaling changes numbers, not engineering meaning
>* Comparable ranges prevent unit-driven model bias

>* Fit scaling on training data only
>* Check outliers before trusting scaled data



In [ ]:
#@title Python Code - Scaling Fundamentals

# Feature scaling changes numerical coordinate systems.
# Engineering units can dominate raw distances.
# Training statistics must scale later data.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create tiny pump measurements with mixed units.
data = pd.DataFrame({
    "flow_lpm": [48, 52, 55, 60, 63, 67],
    "pressure_kpa": [980, 1010, 1040, 1100, 1160, 1210],
    "vibration_g": [0.18, 0.20, 0.23, 0.28, 0.31, 0.35],
})

# Split manually before learning scaling values.
train = data.iloc[:4].copy()
test = data.iloc[4:].copy()

# Learn mean and spread from training data only.
train_mean = train.mean()
train_std = train.std(ddof=0)

# Apply the same transformation to both datasets.
train_scaled = (train - train_mean) / train_std
test_scaled = (test - train_mean) / train_std

# Validate that scaling produced matching shapes.
assert train_scaled.shape == train.shape
assert test_scaled.shape == test.shape

# Compare one raw distance with one scaled distance.
raw_distance = np.linalg.norm(train.iloc[0] - train.iloc[3])
scaled_distance = np.linalg.norm(train_scaled.iloc[0] - train_scaled.iloc[3])

# Show compact results for beginner interpretation.
print("Training rows:", len(train), "Test rows:", len(test))
print("Raw distance between two pumps:", round(raw_distance, 2))
print("Scaled distance between same pumps:", round(scaled_distance, 2))
print("Training scaled means:", train_scaled.mean().round(2).to_dict())
print("Training scaled stds:", train_scaled.std(ddof=0).round(2).to_dict())

# Plot raw and scaled feature ranges side by side.
fig, axes = plt.subplots(1, 2, figsize=(9, 3))

# Raw ranges reveal unit-driven magnitude differences.
train.plot(kind="box", ax=axes[0], title="Raw training features")
axes[0].set_ylabel("Original units")

# Scaled ranges are more comparable numerically.
train_scaled.plot(kind="box", ax=axes[1], title="Scaled training features")
axes[1].set_ylabel("Standardized units")

# Keep labels readable in the single figure.
for axis in axes:
    axis.tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()



### **3.2. Scaling Model Effects**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_02/Lecture_B/image_03_02.jpg?v=1783660102" width="250">



>* Raw units can distort feature importance
>* Scaling helps models learn real patterns

>* Scaling balances distance-based feature comparisons
>* Similarity reflects engineering patterns, not magnitudes

>* Uneven feature ranges slow gradient learning
>* Scaling improves convergence and parameter updates



In [ ]:
#@title Python Code - Scaling Model Effects

# Scaling changes model behavior.
# Distances can be unit dominated.
# Gradients improve after scaling.

import numpy as np
import matplotlib.pyplot as plt

# Create tiny mechanical sensor data.
temperature_c = np.array([48, 50, 52, 80], dtype=float)
pressure_kpa = np.array([500, 520, 540, 560], dtype=float)

# Stack features as observations.
X_raw = np.column_stack((temperature_c, pressure_kpa))
assert X_raw.shape == (4, 2)

# Define a new pump observation.
new_raw = np.array([53, 555], dtype=float)
assert new_raw.size == X_raw.shape[1]

# Compute Euclidean distances manually.
raw_diffs = X_raw - new_raw
raw_distances = np.sqrt(np.sum(raw_diffs**2, axis=1))
raw_nearest = int(np.argmin(raw_distances))

# Scale using training feature statistics.
means = X_raw.mean(axis=0)
stds = X_raw.std(axis=0)
stds = np.where(stds == 0, 1, stds)

# Transform both stored and new observations.
X_scaled = (X_raw - means) / stds
new_scaled = (new_raw - means) / stds
scaled_diffs = X_scaled - new_scaled

# Recompute distances after scaling.
scaled_distances = np.sqrt(np.sum(scaled_diffs**2, axis=1))
scaled_nearest = int(np.argmin(scaled_distances))

# Compare gradient step sizes conceptually.
raw_gradient_sizes = np.abs(X_raw).mean(axis=0)
scaled_gradient_sizes = np.abs(X_scaled).mean(axis=0)

# Print compact teaching results.
print("Raw nearest machine index:", raw_nearest)
print("Scaled nearest machine index:", scaled_nearest)
print("Raw distance range:", round(raw_distances.ptp(), 2))
print("Scaled distance range:", round(scaled_distances.ptp(), 2))
print("Raw average feature magnitudes:", np.round(raw_gradient_sizes, 2))
print("Scaled average feature magnitudes:", np.round(scaled_gradient_sizes, 2))

# Plot raw and scaled distance comparisons.
labels = ["M0", "M1", "M2", "M3"]
x_positions = np.arange(len(labels))
bar_width = 0.35

# Build one compact comparison plot.
plt.figure(figsize=(7, 4))
plt.bar(x_positions - bar_width/2, raw_distances, bar_width, label="Raw")
plt.bar(x_positions + bar_width/2, scaled_distances, bar_width, label="Scaled")

# Label the plot clearly.
plt.xticks(x_positions, labels)
plt.ylabel("Distance to new pump")
plt.title("Scaling changes distance-based similarity")
plt.legend()

# Display the single required plot.
plt.tight_layout()
plt.show()



### **3.3. Scaling Model Effects**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_02/Lecture_B/image_03_03.jpg?v=1783660104" width="250">



>* Different units can distort feature importance
>* Scaling helps models learn real patterns

>* Large-scale features can dominate distance calculations
>* Scaling improves fair similarity-based model decisions

>* Uneven scales can destabilize gradient learning
>* Scaling makes training smoother and more reliable



In [ ]:
#@title Python Code - Scaling Model Effects

# Scaling changes model behavior in engineering datasets.
# This example compares raw and scaled distances.
# It also shows smoother gradient descent.

import numpy as np
import matplotlib.pyplot as plt

# Create tiny pump records with mixed engineering units.
speed_rpm = np.array([1750, 1760, 2600, 2610], dtype=float)
vibration_mm_s = np.array([1.2, 8.5, 1.4, 8.7], dtype=float)

# Stack features into a small design matrix.
X_raw = np.column_stack((speed_rpm, vibration_mm_s))
assert X_raw.shape == (4, 2)

# Define a simple standard scaling function.
def standard_scale(matrix):
    means = matrix.mean(axis=0)
    stds = matrix.std(axis=0)
    return (matrix - means) / stds

# Scale both features to comparable numerical ranges.
X_scaled = standard_scale(X_raw)
assert np.all(np.isfinite(X_scaled))

# Compare distances from pump zero to other pumps.
def distances_from_first(matrix):
    differences = matrix[1:] - matrix[0]
    return np.sqrt((differences ** 2).sum(axis=1))

# Compute raw and scaled geometric distances.
raw_distances = distances_from_first(X_raw)
scaled_distances = distances_from_first(X_scaled)

# Build a tiny gradient descent demonstration.
y = np.array([0.0, 1.0, 0.0, 1.0], dtype=float)
X_design_raw = np.column_stack((np.ones(4), X_raw))
X_design_scaled = np.column_stack((np.ones(4), X_scaled))

# Run fixed-step linear regression gradient descent.
def gradient_descent_losses(design, target, rate, steps):
    weights = np.zeros(design.shape[1])
    losses = []
    for step in range(steps):
        errors = design @ weights - target
        losses.append(float(np.mean(errors ** 2)))
        gradient = (2 / len(target)) * design.T @ errors
        weights = weights - rate * gradient
    return losses

# Use rates that reveal scaling effects clearly.
raw_losses = gradient_descent_losses(X_design_raw, y, 0.0000001, 40)
scaled_losses = gradient_descent_losses(X_design_scaled, y, 0.1, 40)

# Print concise results for interpretation.
print("Raw distances from pump 0:", np.round(raw_distances, 2))
print("Scaled distances from pump 0:", np.round(scaled_distances, 2))
print("Raw loss start/end:", round(raw_losses[0], 3), round(raw_losses[-1], 3))
print("Scaled loss start/end:", round(scaled_losses[0], 3), round(scaled_losses[-1], 3))
print("Scaling prevents large-unit features from dominating distances.")

# Plot loss curves to show optimization behavior.
plt.figure(figsize=(7, 4))
plt.plot(raw_losses, label="raw features")
plt.plot(scaled_losses, label="scaled features")
plt.xlabel("Gradient descent step")
plt.ylabel("Mean squared error")
plt.title("Scaling helps gradient descent move more smoothly")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Features and Splits**</font>


In this lecture, you learned to:
- Extract practical features from tabular, temporal, and image-style mechanical engineering data. 
- Implement manual train and test splitting without using machine learning libraries. 
- Scale numerical features and explain why scaling affects distance-based and gradient-based models. 

In the next Module (Module 3), we will go over 'None'